# ContextCheck: Colab GPU training

Use a GPU runtime for practical training. Upload a source ZIP containing `requirements.txt` and `src/`; model weights and caches are not required. The default pilot uses 20,000 training examples. Set `TRAIN_LIMIT = None` for full training.

In [ ]:
from google.colab import files
from pathlib import Path
import os
import zipfile

uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
project_root = Path('/content/contextcheck')
project_root.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(project_root)
candidates = [path for path in project_root.rglob('requirements.txt') if (path.parent / 'src').is_dir()]
if not candidates:
    raise FileNotFoundError('The uploaded ZIP must contain requirements.txt and src/.')
os.chdir(candidates[0].parent)
print('Project directory:', Path.cwd())


In [ ]:
%pip install -q -r requirements.txt
%pip install -q --upgrade 'protobuf>=6.31.1,<7'
print('Dependencies installed. If Colab requests a restart, restart the runtime and rerun from the project-directory cell.')


In [ ]:
import os
import sys
import torch
import transformers
import datasets
import google.protobuf

os.chdir('/content/contextcheck')
sys.path.insert(0, os.getcwd())
print('Project directory:', os.getcwd())
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__)
print('protobuf:', google.protobuf.__version__)
print('GPU available:', torch.cuda.is_available())


## Inspect and preprocess SNLI

This preserves the official train, validation, and test split boundaries and removes invalid records.

In [ ]:
from src.dataset_loader import load_snli
snli, preprocessing_stats = load_snli()
preprocessing_stats


In [ ]:
import shutil
import subprocess

TRAIN_LIMIT = 20000  # Set to None for the full cleaned training split.
EPOCHS = 1
RESUME_CHECKPOINT = None
python_executable = shutil.which('python')
if python_executable is None:
    raise RuntimeError('Could not locate the active Colab Python executable.')
common = [] if TRAIN_LIMIT is None else ['--max-train-samples', str(TRAIN_LIMIT)]
def run_command(command):
    result = subprocess.run(command, text=True, capture_output=True)
    print('RETURN CODE:', result.returncode)
    print(result.stdout[-12000:])
    print(result.stderr[-12000:])
    if result.returncode != 0:
        raise RuntimeError('Training command failed; inspect the output above.')

run_command([python_executable, '-m', 'src.train_baseline', *common])
transformer_command = [python_executable, '-m', 'src.train_transformer', '--epochs', str(EPOCHS), '--batch-size', '8', '--gradient-accumulation-steps', '2', *common]
if RESUME_CHECKPOINT:
    transformer_command.extend(['--resume-from-checkpoint', RESUME_CHECKPOINT])
run_command(transformer_command)


In [ ]:
from pathlib import Path

model_dir = Path('/content/contextcheck/models/contextcheck_distilbert')
baseline_dir = Path('/content/contextcheck/models/baseline')
evaluation_file = Path('/content/contextcheck/artifacts/evaluation_results.json')
checks = {
    'Transformer model': (model_dir / 'model.safetensors').is_file() or (model_dir / 'pytorch_model.bin').is_file(),
    'Transformer config': (model_dir / 'config.json').is_file(),
    'Transformer tokenizer': (model_dir / 'tokenizer.json').is_file(),
    'Baseline model': (baseline_dir / 'baseline.joblib').is_file(),
    'Evaluation results': evaluation_file.is_file(),
}
for name, present in checks.items():
    print(f'{name}: {present}')
if not all(checks.values()):
    raise FileNotFoundError('One or more expected training artifacts are missing.')


In [ ]:
import shutil
from pathlib import Path

for folder in ('models/contextcheck_distilbert', 'models/baseline', 'artifacts'):
    source = Path('/content/contextcheck') / folder
    archive = shutil.make_archive(f"/content/{folder.replace('/', '_')}", 'zip', source)
    print('Created:', archive)


In [ ]:
from google.colab import files
files.download('/content/models_contextcheck_distilbert.zip')
files.download('/content/models_baseline.zip')
files.download('/content/artifacts.zip')
